In [ ]:
!nvidia-smi

Wed Apr  1 13:02:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder to save your model
import os
os.makedirs('/content/drive/MyDrive/skin_disease_project/checkpoints', exist_ok=True)
print("Drive mounted ✅")

Mounted at /content/drive
Drive mounted ✅


In [ ]:
!pip install timm torch torchvision scikit-learn matplotlib seaborn tqdm -q
print("Dependencies installed ✅")


Dependencies installed ✅


In [ ]:
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)

# Paste your NEW key here after you regenerate it
kaggle_config = {
    "username": "mazbhaul",
    "key": "KGAT_a1191b2d1ceba7f7dc3f53e5d2d5b6f0"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_config, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verify
!cat ~/.kaggle/kaggle.json

# Download
!pip install kaggle -q
!kaggle datasets download -d ismailpromus/skin-diseases-image-dataset --force
print("Download complete ✅")

{"username": "mazbhaul", "key": "KGAT_a1191b2d1ceba7f7dc3f53e5d2d5b6f0"}Dataset URL: https://www.kaggle.com/datasets/ismailpromus/skin-diseases-image-dataset
License(s): copyright-authors
100% 5.19G/5.19G [00:31<00:00, 175MB/s]

Download complete ✅


In [ ]:
!unzip -q skin-diseases-image-dataset.zip -d /content/data/
!ls /content/data/

IMG_CLASSES


In [ ]:
import os

data_dir = "/content/data/IMG_CLASSES"
folders = sorted(os.listdir(data_dir))

total = 0
print("Classes found:\n")
for f in folders:
    path = os.path.join(data_dir, f)
    count = len(os.listdir(path))
    total += count
    print(f"  {f:<45} → {count:,} images")

print(f"\nTotal images: {total:,}")

Classes found:

  1. Eczema 1677                                → 1,677 images
  10. Warts Molluscum and other Viral Infections - 2103 → 2,103 images
  2. Melanoma 15.75k                            → 3,140 images
  3. Atopic Dermatitis - 1.25k                  → 1,257 images
  4. Basal Cell Carcinoma (BCC) 3323            → 3,323 images
  5. Melanocytic Nevi (NV) - 7970               → 7,970 images
  6. Benign Keratosis-like Lesions (BKL) 2624   → 2,079 images
  7. Psoriasis pictures Lichen Planus and related diseases - 2k → 2,055 images
  8. Seborrheic Keratoses and other Benign Tumors - 1.8k → 1,847 images
  9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k → 1,702 images

Total images: 27,153


Training


In [ ]:
%%writefile train.py

import os
import csv
import json
import argparse
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from tqdm import tqdm

# ── Class map ──
CLASS_FOLDER_MAP = {
    "1":  "Eczema",
    "2":  "Melanoma",
    "3":  "Atopic Dermatitis",
    "4":  "Basal Cell Carcinoma",
    "5":  "Melanocytic Nevi",
    "6":  "Benign Keratosis-like Lesions",
    "7":  "Psoriasis",
    "8":  "Seborrheic Keratoses",
    "9":  "Tinea Ringworm Candidiasis",
    "10": "Warts Molluscum and other Viral Infections",
}
CLASS_NAMES = list(CLASS_FOLDER_MAP.values())


def get_label_from_folder(folder_name):
    prefix = folder_name.split(".")[0].strip()
    return CLASS_FOLDER_MAP.get(prefix, None)


def build_file_list(data_dir):
    samples = []
    for folder in os.listdir(data_dir):
        label = get_label_from_folder(folder)
        if label is None:
            continue
        label_idx = CLASS_NAMES.index(label)
        folder_path = os.path.join(data_dir, folder)
        for fname in os.listdir(folder_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                samples.append((os.path.join(folder_path, fname), label_idx))
    return samples


class SkinDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


def get_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2,
                               saturation=0.2, hue=0.1),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((300, 300)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


def get_dataloaders(data_dir, batch_size):
    all_samples = build_file_list(data_dir)
    labels = [s[1] for s in all_samples]

    train_val, test = train_test_split(
        all_samples, test_size=0.1, stratify=labels, random_state=42
    )
    labels_tv = [s[1] for s in train_val]
    train, val = train_test_split(
        train_val, test_size=0.111, stratify=labels_tv, random_state=42
    )

    train_tf, val_tf = get_transforms()

    class_counts = np.bincount([s[1] for s in train], minlength=10)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = [class_weights[s[1]] for s in train]
    sampler = WeightedRandomSampler(sample_weights, len(train), replacement=True)

    train_loader = DataLoader(SkinDataset(train, train_tf), batch_size=batch_size,
                              sampler=sampler, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(SkinDataset(val, val_tf),   batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(SkinDataset(test, val_tf),  batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
    return train_loader, val_loader, test_loader


# ── PDF REQUIREMENT: Confusion matrix + classification report ──
def evaluate_and_save(model, test_loader, device, output_dir):
    print("\n📊 Running evaluation on test set...")
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(test_loader, desc="Evaluating"):
            imgs = imgs.to(device)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    # Overall accuracy
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"\n✅ Test Accuracy: {accuracy:.2%}")

    # Classification report
    report = classification_report(
        all_labels, all_preds,
        target_names=CLASS_NAMES,
        output_dict=True
    )
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

    # Save report as JSON
    report["test_accuracy"] = accuracy
    with open(os.path.join(output_dir, "evaluation_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    # Confusion matrix PNG
    cm = confusion_matrix(all_labels, all_preds)
    short_names = [
        "Eczema", "Melanoma", "Atopic D.",
        "Basal CC", "Mel. Nevi", "Benign K.",
        "Psoriasis", "Seborrh.", "Tinea", "Warts"
    ]
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short_names, yticklabels=short_names)
    plt.title("Confusion Matrix — Skin Disease Classifier")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    cm_path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=150)
    plt.show()
    print(f"Confusion matrix saved → {cm_path}")

    return accuracy


def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    os.makedirs(args.output_dir, exist_ok=True)
    log_path = os.path.join(args.output_dir, "training_log.csv")

    model = timm.create_model("efficientnet_b3", pretrained=True, num_classes=10)
    model = model.to(device)

    train_loader, val_loader, test_loader = get_dataloaders(
        args.data_dir, args.batch_size
    )

    # ── Freeze backbone initially ──
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.lr, weight_decay=1e-2
    )
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=args.epochs
    )

    # ── Unfreeze epoch: 3 for <=10 epochs, 6 for >10 epochs ──
    unfreeze_at = 3 if args.epochs <= 10 else 6

    best_val_loss = float("inf")
    patience_counter = 0
    best_path = os.path.join(args.output_dir, "best_model.pt")

    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(
            ["epoch", "train_loss", "val_loss", "val_accuracy", "lr"]
        )

    print(f"\n🚀 Starting training (unfreezing all layers at epoch {unfreeze_at})...\n")

    for epoch in range(1, args.epochs + 1):

        if epoch == unfreeze_at:
            print("🔓 Unfreezing all layers for fine-tuning...")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=args.lr * 0.1, weight_decay=1e-2
            )

        # ── Train ──
        model.train()
        train_loss = 0.0
        for imgs, labels in tqdm(train_loader,
                                  desc=f"Epoch {epoch}/{args.epochs} [Train]"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validate ──
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader,
                                      desc=f"Epoch {epoch}/{args.epochs} [Val]  "):
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                val_loss += criterion(out, labels).item()
                correct += (out.argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_loss /= len(val_loader)
        val_acc = correct / total

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"\n  Epoch {epoch:02d} → train_loss: {train_loss:.4f} | "
              f"val_loss: {val_loss:.4f} | "
              f"val_acc: {val_acc:.2%} | lr: {current_lr:.2e}\n")

        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow(
                [epoch, train_loss, val_loss, val_acc, current_lr]
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_accuracy": val_acc,
                "val_loss": val_loss,
                "class_names": CLASS_NAMES,
            }, best_path)
            print(f"  ✅ Best model saved (val_loss={val_loss:.4f})\n")
        else:
            patience_counter += 1
            print(f"  No improvement. Patience: {patience_counter}/{args.patience}\n")
            if patience_counter >= args.patience:
                print("⏹ Early stopping triggered.")
                break

    # ── PDF REQUIREMENT: Run evaluation after training ──
    checkpoint = torch.load(best_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    final_accuracy = evaluate_and_save(model, test_loader, device, args.output_dir)

    print(f"\n🎉 Training complete!")
    print(f"   Best val_loss  : {best_val_loss:.4f}")
    print(f"   Test accuracy  : {final_accuracy:.2%}")
    print(f"   Model saved    : {best_path}")
    print(f"   Confusion matrix + report saved to {args.output_dir}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir",   default="/content/data/IMG_CLASSES")
    parser.add_argument("--epochs",     type=int,   default=10)
    parser.add_argument("--batch_size", type=int,   default=32)
    parser.add_argument("--lr",         type=float, default=1e-4)
    parser.add_argument("--patience",   type=int,   default=3)
    parser.add_argument("--output_dir",
        default="/content/drive/MyDrive/skin_disease_project/checkpoints")
    args = parser.parse_args()
    train(args)

Overwriting train.py


In [ ]:
!python train.py \
    --data_dir /content/data/IMG_CLASSES \
    --epochs 10 \
    --batch_size 32 \
    --patience 3 \
    --output_dir /content/drive/MyDrive/skin_disease_project/checkpoints

Using device: cuda
model.safetensors: 100% 49.3M/49.3M [00:00<00:00, 54.2MB/s]
Train: 21,724 | Val: 2,713 | Test: 2,716

🚀 Starting training (unfreezing all layers at epoch 3)...

Epoch 1/10 [Train]: 100% 679/679 [06:48<00:00,  1.66it/s]
Epoch 1/10 [Val]  : 100% 85/85 [00:30<00:00,  2.74it/s]

  Epoch 01 → train_loss: 2.8523 | val_loss: 2.3590 | val_acc: 24.00% | lr: 9.76e-05

  ✅ Best model saved (val_loss=2.3590)

Epoch 2/10 [Train]: 100% 679/679 [06:39<00:00,  1.70it/s]
Epoch 2/10 [Val]  : 100% 85/85 [00:28<00:00,  2.93it/s]

  Epoch 02 → train_loss: 2.2046 | val_loss: 2.2204 | val_acc: 34.28% | lr: 9.05e-05

  ✅ Best model saved (val_loss=2.2204)

🔓 Unfreezing all layers for fine-tuning...
Epoch 3/10 [Train]: 100% 679/679 [08:22<00:00,  1.35it/s]
Epoch 3/10 [Val]  : 100% 85/85 [00:28<00:00,  2.96it/s]

  Epoch 03 → train_loss: 1.6473 | val_loss: 1.3309 | val_acc: 52.16% | lr: 1.00e-05

  ✅ Best model saved (val_loss=1.3309)

Epoch 4/10 [Train]: 100% 679/679 [08:20<00:00,  1.36it/s]